In [ ]:
from pyspark.sql.functions import *

df = spark.read.table("finops_catalog_2.finops.bronze_cloud_budget")

In [ ]:

df_clean = (
    df
    .dropDuplicates()
    .filter(col("date").isNotNull())
    .filter(col("net_cost").isNotNull())
    .filter(col("budget_amount").isNotNull())
    .filter(col("usage_quantity").isNotNull())
)

df_transformed = (
    df_clean
    .withColumn("business_unit", trim(initcap(col("business_unit"))))
    .withColumn("department", trim(initcap(col("department"))))
    .withColumn("service", trim(upper(col("service"))))
    .withColumn("cloud_provider", trim(upper(col("cloud_provider"))))
    .withColumn("budget_status", trim(initcap(col("budget_status"))))
)

In [ ]:
spark.sql("DROP TABLE IF EXISTS finops_catalog_2.finops.silver_cloud_budget")

df_transformed.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("finops_catalog_2.finops.silver_cloud_budget")